# EDA — IEEE-CIS Fraud Detection Dataset
**590K transactions, 434 features, 3.5% fraud rate**

## 1. Data Loading

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/graduation-p'):
        !git clone https://github.com/berk0vic/graduation-p.git /content/graduation-p
    os.chdir('/content/graduation-p/notebooks')
    sys.path.insert(0, '/content/graduation-p')
    !rm -rf /content/graduation-p/data
    !ln -s /content/drive/MyDrive/graduation-p/data /content/graduation-p/data
    !pip install -q torch-scatter torch-sparse torch-geometric xgboost shap -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
else:
    sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.ieee_cis_loader import load_raw, get_label_column

train_df, test_df = load_raw()
label = get_label_column()
print(f"Shape: {train_df.shape}")
train_df[label].value_counts(normalize=True)

## 2. Missing Value Analysis

In [ ]:
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).sort_values(ascending=False)
print("Total features:", len(train_df.columns))
print("\nFeatures with >50% missing:", (missing_pct > 50).sum())
print("\nTop 20 most missing features:")
missing_pct.head(20)

## 3. Transaction Amount Distribution (Fraud vs Normal)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df[train_df[label]==0]['TransactionAmt'], bins=50, alpha=0.7,
             label='Normal', color='blue')
axes[0].hist(train_df[train_df[label]==1]['TransactionAmt'], bins=50, alpha=0.7,
             label='Fraud', color='red')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()

axes[1].hist(train_df[train_df[label]==0]['TransactionAmt'], bins=50, alpha=0.7,
             label='Normal', color='blue', log=True)
axes[1].hist(train_df[train_df[label]==1]['TransactionAmt'], bins=50, alpha=0.7,
             label='Fraud', color='red', log=True)
axes[1].set_title('Transaction Amount (Log Scale)')
axes[1].set_xlabel('Amount ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Normal avg: ${train_df[train_df[label]==0]['TransactionAmt'].mean():.2f}")
print(f"Fraud  avg: ${train_df[train_df[label]==1]['TransactionAmt'].mean():.2f}")

## 4. Fraud Rate by Category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fraud_by_product = train_df.groupby('ProductCD')['isFraud'].mean() * 100
fraud_by_product.sort_values().plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title('Fraud Rate by Product Category (%)')
axes[0].set_ylabel('Fraud %')
axes[0].tick_params(axis='x', rotation=0)

fraud_by_card4 = train_df.groupby('card4')['isFraud'].mean() * 100
fraud_by_card4.sort_values().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Fraud Rate by Card Brand (%)')
axes[1].set_ylabel('Fraud %')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 5. Feature Categorization by Missing Rate

In [ ]:
low_missing = missing_pct[missing_pct < 30].index.tolist()
high_missing = missing_pct[(missing_pct >= 30) & (missing_pct < 80)].index.tolist()
drop_cols = missing_pct[missing_pct >= 80].index.tolist()

print(f"Keep (<30% missing): {len(low_missing)} features")
print(f"Fill (30-80% missing): {len(high_missing)} features")
print(f"Drop (>80% missing): {len(drop_cols)} features")
print(f"\nDropped columns sample: {drop_cols[:10]}")

## 6. Data Cleaning — Drop >80% Missing, Median/Mode Fill

In [ ]:
train_clean = train_df.drop(columns=drop_cols).copy()

num_cols = [c for c in train_clean.select_dtypes(include=['float64', 'int64']).columns if c != 'isFraud']
cat_cols = train_clean.select_dtypes(include=['str']).columns.tolist()

for col in num_cols:
    train_clean[col] = train_clean[col].fillna(train_clean[col].median())

for col in cat_cols:
    train_clean[col] = train_clean[col].fillna(train_clean[col].mode()[0])

print(f"Clean dataset shape: {train_clean.shape}")
print(f"Remaining missing values: {train_clean.isnull().sum().sum()}")

## 7. Export Cleaned Dataset

In [ ]:
train_clean.to_csv('../data/processed/ieee_cis_train_clean.csv', index=False)
print("Saved!")

## 8. Heterogeneous Graph Construction (`build_hetero_graph`)

In [ ]:
import time
import torch
from src.graph.builder import build_hetero_graph

FAST_SAMPLE = True
SAMPLE_FRAC = 0.03

df_graph = train_df.copy()
if FAST_SAMPLE:
    df_graph = df_graph.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)

t0 = time.time()
data = build_hetero_graph(df_graph, dataset='ieee_cis')
print(f"Build time: {time.time() - t0:.1f}s")

y_t = data['transaction'].y
n_fraud = int(y_t.sum())
n_all = len(y_t)
print(f"\nTransactions: {n_all:,} | Fraud: {n_fraud:,} ({100*n_fraud/n_all:.2f}%)")
print(f"Feature dim (txn): {data['transaction'].x.shape[1]}")
print(f"Account nodes: {data['account'].num_nodes:,} | Merchant: {data['merchant'].num_nodes:,}")
print("\nEdge types:")
for et in data.edge_types:
    ei = data[et].edge_index
    print(f"  {et}: {ei.shape[1]:,} edges")

### Node & Edge Types

| Node | Meaning | Features |
|------|---------|----------|
| **transaction** | Single transaction row | velocity + numerical + one-hot + IQR |
| **account** | `card1` proxy | Card-level amount statistics |
| **merchant** | `addr1` + `ProductCD` | Address+product group statistics |

**Edges:** account→transaction, transaction→merchant, temporal (same card), shared identity/device

### Subgraph Visualization (NetworkX)

In [ ]:
import networkx as nx

def _nid(prefix, i):
    return f"{prefix}:{int(i)}"

y_np = data['transaction'].y.numpy()
f_idx = np.where(y_np == 1)[0]
n_idx = np.where(y_np == 0)[0]
rng = np.random.default_rng(42)
seeds = []
if len(f_idx):
    seeds.extend(rng.choice(f_idx, size=min(6, len(f_idx)), replace=False).tolist())
if len(n_idx):
    seeds.extend(rng.choice(n_idx, size=min(6, len(n_idx)), replace=False).tolist())
seeds = list(dict.fromkeys(seeds))

ei_a = data['account', 'initiates', 'transaction'].edge_index.numpy()
acct_of = {ei_a[1, i]: ei_a[0, i] for i in range(ei_a.shape[1])}
ei_m = data['transaction', 'paid_to', 'merchant'].edge_index.numpy()
merch_of = {ei_m[0, i]: ei_m[1, i] for i in range(ei_m.shape[1])}

txn_set = set(seeds)
for t in list(txn_set):
    if t in acct_of:
        a = acct_of[t]
        mask = ei_a[0] == a
        for t2 in ei_a[1, mask][:12]:
            txn_set.add(int(t2))
acct_set = {acct_of[t] for t in txn_set if t in acct_of}
merch_set = {merch_of[t] for t in txn_set if t in merch_of}

G = nx.MultiDiGraph()
for t in txn_set:
    fr = y_np[t] == 1
    G.add_node(_nid('t', t), kind='txn', fraud=bool(fr), label=f"T{t}")
for a in acct_set:
    G.add_node(_nid('a', a), kind='acct', label=f"A{a}")
for m in merch_set:
    G.add_node(_nid('m', m), kind='merch', label=f"M{m}")

def add_et(etype, lab, src_fn, dst_fn):
    ei = data[etype].edge_index.numpy()
    for i in range(ei.shape[1]):
        s, d = int(ei[0, i]), int(ei[1, i])
        u, v = src_fn(s, d), dst_fn(s, d)
        if u in G and v in G:
            G.add_edge(u, v, label=lab)

add_et(('account', 'initiates', 'transaction'), 'initiates', lambda s,d: _nid('a', s), lambda s,d: _nid('t', d))
add_et(('transaction', 'paid_to', 'merchant'), 'paid_to', lambda s,d: _nid('t', s), lambda s,d: _nid('m', d))
if ('transaction', 'followed_by', 'transaction') in data.edge_types:
    add_et(('transaction', 'followed_by', 'transaction'), 'time', lambda s,d: _nid('t', s), lambda s,d: _nid('t', d))
if ('transaction', 'shares_identity', 'transaction') in data.edge_types:
    add_et(('transaction', 'shares_identity', 'transaction'), 'id', lambda s,d: _nid('t', s), lambda s,d: _nid('t', d))

pos = nx.spring_layout(G, seed=42, k=0.35, iterations=60)
plt.figure(figsize=(12, 8))
colors = []
for n, attr in G.nodes(data=True):
    if attr.get('kind') == 'acct':
        colors.append('#2ecc71')
    elif attr.get('kind') == 'merch':
        colors.append('#9b59b6')
    elif attr.get('fraud'):
        colors.append('#e74c3c')
    else:
        colors.append('#3498db')
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=380, alpha=0.9)
nx.draw_networkx_edges(G, pos, alpha=0.35, arrows=True, arrowsize=12, width=0.8)
nx.draw_networkx_labels(G, pos, labels={n: d.get('label', '') for n, d in G.nodes(data=True)}, font_size=7)
plt.title('Sample Heterogeneous Subgraph (transaction / account / merchant)')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

## 10. Tabular Features → XGBoost Quick Validation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

X = data['transaction'].x.numpy()
y_bin = data['transaction'].y.numpy()
print(f"Tabular X shape: {X.shape}")

idx = np.arange(len(y_bin))
tr_i, te_i = train_test_split(idx, test_size=0.25, stratify=y_bin, random_state=42)
if len(tr_i) > 8000: tr_i = tr_i[:8000]
if len(te_i) > 2000: te_i = te_i[:2000]

ratio = y_bin[tr_i].sum() / max(1, (len(tr_i) - y_bin[tr_i].sum()))
xgb = XGBClassifier(
    n_estimators=120, max_depth=5, learning_rate=0.08,
    scale_pos_weight=float(1.0 / ratio),
    eval_metric='auc', tree_method='hist', n_jobs=-1, random_state=42, verbosity=0,
)
xgb.fit(X[tr_i], y_bin[tr_i])
auc = roc_auc_score(y_bin[te_i], xgb.predict_proba(X[te_i])[:, 1])
print(f"Mini-XGBoost ROC-AUC: {auc:.4f}")

## 11. Save Processed Graph

In [ ]:
import os
os.makedirs('../data/processed/ieee_cis', exist_ok=True)
out_pt = '../data/processed/ieee_cis/hetero_graph_eda_sample.pt'
torch.save(data, out_pt)
print(f"Saved: {out_pt}")